Cell 1: setup and load

In [1]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_score, recall_score, f1_score, confusion_matrix)

load_dotenv(find_dotenv())
engine = create_engine(URL.create(
    "postgresql+psycopg2",
    username=os.getenv("PG_USER"), password=os.getenv("PG_PASSWORD"),
    host=os.getenv("PG_HOST"), port=int(os.getenv("PG_PORT")),
    database=os.getenv("PG_DB"),
))

df = pd.read_sql("""
    SELECT f.*, l.target_capped_holdout
    FROM features.user_features f
    JOIN features.user_labels l USING (user_id)
""", engine)

TARGET = "target_capped_holdout"
FEATURES = [c for c in df.columns if c not in ("user_id", TARGET)]

# Leakage guard: fail loudly if a holdout-derived column sneaks in
banned = ("holdout", "z_holdout", "label_", "eval_set")
assert not any(b in c for c in FEATURES for b in banned), "Leakage: holdout-derived column in features"

print(df.shape, "| base rate:", round(df[TARGET].mean(), 3))
print(len(FEATURES), "features:", FEATURES)

(206209, 22) | base rate: 0.306
20 features: ['n_prior_orders', 'n_gaps', 'tenure_days_lb', 'mean_gap', 'median_gap', 'sd_gap', 'mean_gap_uncapped', 'n_capped_gaps', 'pct_capped_gaps', 'is_censored_heavy', 'n_same_day_gaps', 'last_prior_gap', 'recent_mean_gap', 'gap_trend', 'avg_basket_size', 'reorder_ratio', 'distinct_products', 'variety_ratio', 'avg_order_hour', 'is_truncated_history']


Cell 2: stratified split

In [2]:
train, test = train_test_split(
    df, test_size=0.2, stratify=df[TARGET], random_state=42
)
y_train, y_test = train[TARGET], test[TARGET]
print(len(train), len(test), round(y_train.mean(), 3), round(y_test.mean(), 3))

164967 41242 0.306 0.306


Cell 3: baseline scoring helper

In [4]:
def report(name, y_true, y_pred, y_score=None):
    out = {
        "model": name,
        "accuracy": (y_true == y_pred).mean(),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    if y_score is not None:
        out["roc_auc"] = roc_auc_score(y_true, y_score)
        out["pr_auc"] = average_precision_score(y_true, y_score)
    return out

Cell 4: two baselines on the test set

In [5]:
results = []

# Baseline 1: always predict "not late"
results.append(report("majority (all 0)", y_test, np.zeros(len(y_test), dtype=int),
                      np.zeros(len(y_test))))

# Baseline 2: "predict capped if the last prior gap was capped"
naive_pred = (test["last_prior_gap"] == 30).astype(int)
results.append(report("naive: last gap capped", y_test, naive_pred, test["last_prior_gap"]))

pd.DataFrame(results).round(3)

,model,accuracy,precision,recall,f1,roc_auc,pr_auc
0,majority (all 0),0.694,0.000,0.000,0.000,0.500,0.306
1,naive: last gap capped,0.702,0.516,0.434,0.471,0.685,0.458


Cell 5: the confusion matrix of the naive rule

In [6]:
print(pd.DataFrame(confusion_matrix(y_test, naive_pred),
                   index=["actual on-time", "actual capped"],
                   columns=["pred on-time", "pred capped"]))

                pred on-time  pred capped
actual on-time         23447         5158
actual capped           7148         5489
